In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

'''
2. Projektowanie i trenowanie sieci neuronowej
   Zaproponuj optymalną sieć neuronową aproksymującą wczytane dane.
   Napisz własną klasę sieci neuronowej korzystając z biblioteki numpy.
   Użyj nieliniowej funkcji aktywacji w warstwie ukrytej (np. tanh lub sigmoidalna).
   Wytrenuj sieć metodą wsadową z użyciem propagacji wstecznej błędu.

5. Funkcja aktywacji RELU
   Zaproponuj sieć z funkcją aktywacji RELU w warstwie ukrytej.
   Wytrenuj sieć metodą wsadową z propagacją wsteczną błędu.

   Siec uzywa funkcji aktywacji zaleznie od wartosci parametru function
'''

class NeutralNetworkRegression:
  def __init__(self, input_size, hidden_size, output_size, function):
    #initialization of weights, biases and metric history (MSE and R^2)
    self.input_size = input_size
    self.hidden_size = hidden_size
    self.output_size = output_size

    # Weights initialisation
    self.weights_input_hidden = np.random.randn(input_size, hidden_size) * np.sqrt(2 / input_size)
    self.weights_hidden_output = np.random.randn(hidden_size, output_size) * np.sqrt(2 / hidden_size)

    #Bias initialization for hidden and output layers
    self.bias_hidden = np.zeros(hidden_size)
    self.bias_output = np.zeros(output_size)

    #History storage for evaluation metrics
    self.history_mse = []
    self.history_r2 = []

  def tanh(self, x):
    return np.tanh(x)

  def tanh_derivative(self, x):# derivative 1-tanh(x)^2
    return 1.0 - np.tanh(x)**2

  def relu(self,x):
    return np.maximum(0,1*x)

  def relu_derivative(self,x):
    return np.where(x > 0, 1, np.where(x<0, 0, 0.5))

  def forward(self, X, function):
    hidden_input = np.dot(X, self.weights_input_hidden) + self.bias_hidden

    if function == "relu":
      hidden_output = self.relu(hidden_input)
    elif function == "tanh":
      hidden_output = self.tanh(hidden_input)

    output_input = np.dot(hidden_output, self.weights_hidden_output) + self.bias_output

    if function == "relu":
      return self.relu(output_input)
    elif function == "tanh":
      return self.tanh(output_input)

  def backward(self, X, y, output, learning_rate, reg_lambda, function):
    output_error = y-output

    if function == "relu":
      hidden_output = self.relu(np.dot(X, self.weights_input_hidden) + self.bias_hidden)
    elif function == "tanh":
      hidden_output = self.tanh(np.dot(X, self.weights_input_hidden) + self.bias_hidden)

    gradient_hidden_output = np.dot(hidden_output.T, output_error)

    hidden_error = np.dot(output_error, self.weights_hidden_output.T)

    if function == "relu":
      hidden_error *= self.relu_derivative(hidden_output)
    elif function == "tanh":
      hidden_error *= self.tanh_derivative(hidden_output)

    gradient_input_hidden = np.dot(X.T, hidden_error)

    # weight and bias updates
    self.weights_hidden_output += learning_rate * (gradient_hidden_output - reg_lambda * self.weights_hidden_output)
    self.weights_input_hidden += learning_rate * (gradient_input_hidden - reg_lambda * self.weights_input_hidden)

    self.bias_output += np.sum(output_error, axis=0) * learning_rate
    self.bias_hidden += np.sum(hidden_error, axis=0) * learning_rate

  def train(self, X, y, epochs, learning_rate, reg_lambda, function):
    print("Start training neutral networks")

    for epoch in range(1, epochs):
      output = self.forward(X, function)
      self.backward(X, y, output, learning_rate, reg_lambda, function)

      mse = mean_squared_error(y, output)
      r2 = r2_score(y, output)

      self.history_mse.append(mse)
      self.history_r2.append(r2)

      if epoch %(epochs//10) == 0:
        print(f"[%] {epoch}/{epochs} MSE: {mse:.4f} R^2: {r2:.4f}")

      if r2 >= 0.95: #early stopping
        print("Training stopped, R^2 score reached 0.95")
        break

  '''
  4. Zmiana sposobu trenowania
      Zmień sposób trenowania z wsadowego na online (stochastyczny).
      Wytrenuj sieć i oceń jej działanie.
  '''

  def train_online(self, X, y, epochs, learning_rate, reg_lambda, function):
    print("Start training (ONLINE)")

    n = X.shape[0]

    for epoch in range(epochs):
        for i in range(n):
            xi = X[i:i+1]   # jedna próbka
            yi = y[i:i+1]

            output = self.forward(xi, function)
            self.backward(xi, yi, output, learning_rate, reg_lambda, function)

        # ocena po epoce
        output_all = self.forward(X, function)

        mse = mean_squared_error(y, output_all)
        r2 = r2_score(y, output_all)

        self.history_mse.append(mse)
        self.history_r2.append(r2)

        if epoch % (epochs//10) == 0:
            print(f"[ONLINE] {epoch}/{epochs} MSE: {mse:.4f} R^2: {r2:.4f}")

# Plot result function
def plotResults(final_mse, final_r2, history_mse, history_r2, hidden_size, learning_rate):
  plt.rcParams.update({'font.size': 14})
  plt.figure(figsize=(15, 10))

  plt.subplot(2, 2, 1)
  plt.scatter(range(len(history_r2)), history_mse, marker = '.')
  plt.ylabel("MSE")
  plt.title(f"MSE vs. Epochs, Final MSE: {round(final_mse, 3)}")
  plt.ylim(0,0.2)

  plt.subplot(2, 2, 2)
  plt.scatter(range(len(history_r2)), history_r2, marker = '.')
  plt.xlabel("Epochs")
  plt.ylabel("R^2")
  plt.title(f"R^2 vs. Epochs, Final R^2: {round(final_r2, 2)}")
  plt.ylim(0,1)

  plt.tight_layout()

  plt.subplot(2, 2, 3)
  plt.scatter(X_train, y_train, color = "blue", label="Training Data")
  plt.scatter(X_test, y_test, color = "red", label="Test Data")
  plt.scatter(X_train, output_train, color = "green", label="Predictions", marker="s")
  plt.scatter(X_test, output_test, color = "green", label="Predictions", marker="x")

  plt.xlabel("X")
  plt.ylabel("Y")
  plt.title(f"Neutrons = {hidden_size}, Epochs={len(history_r2)}, learn_r={learning_rate}")
  plt.legend()


'''
1. Wczytanie i podział danych
   Wczytaj dane z pliku `daneXX.txt`. Wybierz dane z folderu public/mmajew/MIW/07 NN/Dane
   Zaproponuj i zrealizuj podział danych na zbiory treningowy i testowy.

   XX to plik ktory wczytujemy.
   Dane podzielilem: 70% treningowe 30% testowe
'''

XX = 9
#Load dataset from file
with open(f"/content/drive/MyDrive/Colab Notebooks/MIW_Project_3/Dane/dane{XX}.txt","r") as file:
  data = file.readlines()

# Parse input-output pairs
x_data, y_data = zip(*[map(float, line.split()) for line in data])

# Data normalization
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()
X_dataNormalized = scaler_x.fit_transform(np.array(x_data).reshape(-1,1))
y_dataNormalized = scaler_y.fit_transform(np.array(y_data).reshape(-1,1))


# RELU ---------------------------------
print("\nRELU:")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_dataNormalized,
    y_dataNormalized,
    test_size=0.3, # podzial 70/30
    random_state=42
)

# Initialize and train neutral network
input_size = 1
hidden_size = 300
output_size = 1
epochs = 2000
learning_rate = 0.005

nn = NeutralNetworkRegression(input_size, hidden_size, output_size, "relu")
nn.train(
    X_train,
    y_train,
    epochs,
    learning_rate,
    reg_lambda=0.01,
    function = "relu"
)

#predictions
output_train = nn.forward(X_train, "relu")
output_test = nn.forward(X_test, "relu")

#final metrics
final_mse = nn.history_mse[-1]
final_r2 = nn.history_r2[-1]

'''
3. Ocena działania sieci – underfitting/overfitting
   Oceń działanie sieci pod kątem dopasowania: niedopasowanie, dopasowanie optymalne, zbytnie dopasowanie.
'''

train_r2 = r2_score(y_train, output_train)
test_r2 = r2_score(y_test, output_test)

print("Train R2:", train_r2)
print("Test R2:", test_r2)

plotResults(final_mse, final_r2, nn.history_mse, nn.history_r2, hidden_size, learning_rate)


# TANH ---------------------------------
print("\nTANH:")

#train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_dataNormalized,
    y_dataNormalized,
    test_size=0.3,
    random_state=42
)

#initialize and train neutral network
input_size = 1
hidden_size = 300
output_size = 1
epochs = 2000
learning_rate = 0.005

nn = NeutralNetworkRegression(input_size, hidden_size, output_size, "tanh")
nn.train(
    X_train,
    y_train,
    epochs,
    learning_rate,
    reg_lambda=0.01,
    function = "tanh"
)

#predictions
output_train = nn.forward(X_train, "tanh")
output_test = nn.forward(X_test, "tanh")

#final metrics
final_mse = nn.history_mse[-1]
final_r2 = nn.history_r2[-1]

train_r2 = r2_score(y_train, output_train)
test_r2 = r2_score(y_test, output_test)

print("Train R2:", train_r2)
print("Test R2:", test_r2)

plotResults(final_mse, final_r2, nn.history_mse, nn.history_r2, hidden_size, learning_rate)


# ONLINE ---------------------------------
print("\nONLINE:")

#train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_dataNormalized,
    y_dataNormalized,
    test_size=0.3,
    random_state=42
)

#initialize and train neutral network
input_size = 1
hidden_size = 300
output_size = 1
epochs = 2000
learning_rate = 0.005

nn = NeutralNetworkRegression(input_size, hidden_size, output_size, "relu")
nn.train_online(
    X_train,
    y_train,
    epochs,
    learning_rate,
    reg_lambda=0.01,
    function = "relu"
)

#predictions
output_train = nn.forward(X_train, "tanh")
output_test = nn.forward(X_test, "tanh")

#final metrics
final_mse = nn.history_mse[-1]
final_r2 = nn.history_r2[-1]

train_r2 = r2_score(y_train, output_train)
test_r2 = r2_score(y_test, output_test)

print("Train R2:", train_r2)
print("Test R2:", test_r2)

plotResults(final_mse, final_r2, nn.history_mse, nn.history_r2, hidden_size, learning_rate)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/MIW_Project_3/Dane/dane9.txt'